# 🧠 Dataset 9: Meningitis con Valores Faltantes
**Propósito:** Clasificación clínica — Predecir diagnóstico o nivel de riesgo de meningitis.  
**Desafío principal:** Datos médicos con missingness estructural — los médicos no realizan todas las pruebas a todos los pacientes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos

In [ ]:
df = pd.read_csv('../datasets/4 data sets/dataset9/mening missing 12.csv')
print(f"Shape: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")
df.head(3)

## 2. Análisis Exhaustivo de Missingness
> **Tipos de datos faltantes en medicina:**
> - **MCAR:** Falta completamente al azar (ej: muestra rota en laboratorio)
> - **MAR:** Falta dependiendo de otras variables observadas
> - **MNAR:** Falta por la propia gravedad del dato (ej: no se hace punción lumbar si el paciente está muy estable o muy grave)

In [ ]:
nulos = df.isnull().sum()
pct   = (nulos/len(df)*100).round(2)
resumen = pd.DataFrame({'Nulos': nulos, '%': pct}).sort_values('%', ascending=False)

print("=== ANÁLISIS DE MISSINGNESS ===")
print(resumen[resumen['Nulos']>0].to_string())

plt.figure(figsize=(12,5))
pct_nonzero = pct[pct>0].sort_values(ascending=False)
colors = ['tomato' if p > 30 else 'orange' if p > 10 else 'steelblue' for p in pct_nonzero]
pct_nonzero.plot(kind='bar', color=colors)
plt.title('% de Valores Faltantes por Columna\n(Rojo >30%, Naranja >10%, Azul ≤10%)')
plt.ylabel('%')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('mening_nulos.png', dpi=100)
plt.show()

## 3. Análisis de Patrón de Missingness (Heatmap)

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mapa de Valores Faltantes por Paciente y Variable\n(Amarillo = faltante, Morado = presente)')
plt.xlabel('Variable')
plt.tight_layout()
plt.savefig('mening_patron.png', dpi=100)
plt.show()
print("\nSi los NaN forman PATRONES no aleatorios → MNAR (los más difíciles de manejar)")

## 4. EDA — Variables Clínicas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribución de edad
df['Age'].hist(ax=axes[0,0], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribución de Edad')
axes[0,0].set_xlabel('Años')

# Diagnóstico
if 'Diagnosis' in df.columns:
    df['Diagnosis'].value_counts().plot(kind='bar', ax=axes[0,1], color='coral')
    axes[0,1].set_title('Distribución de Diagnóstico')
    axes[0,1].tick_params(axis='x', rotation=30)

# Variables LCR (Líquido Cefalorraquídeo) — las más importantes
lcr_vars = [c for c in ['Protein_Level','Glucose_Level','WBC_Count'] if c in df.columns]
for i, col in enumerate(lcr_vars[:2]):
    ax = axes[1][i]
    df[col].dropna().hist(ax=ax, bins=30, color='seagreen', edgecolor='white')
    ax.set_title(f'Distribución de {col}')
    ax.set_xlabel('Valor')

plt.tight_layout()
plt.savefig('mening_eda.png', dpi=100)
plt.show()

## 5. Estrategia de Imputación: KNN vs Media
> En datos médicos, la **imputación por media** puede destruir la validez clínica. El **KNN Imputer** preserva relaciones multivariadas.

In [ ]:
# Separar variables a imputar
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()

df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    mask = df_enc[col].notna()
    if mask.sum() > 0:
        le.fit(df_enc[col].dropna().astype(str))
        df_enc.loc[mask, col] = le.transform(df_enc[col][mask].astype(str))

# KNN Imputer
knn_imputer = KNNImputer(n_neighbors=7)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_enc.apply(pd.to_numeric, errors='coerce')),
                      columns=df_enc.columns)

print(f"Nulos tras KNN Imputer: {df_knn.isnull().sum().sum()}")

# Media Imputer (para comparación)
mean_imputer = SimpleImputer(strategy='mean')
df_mean = pd.DataFrame(mean_imputer.fit_transform(df_enc.apply(pd.to_numeric, errors='coerce')),
                       columns=df_enc.columns)

print(f"Nulos tras Mean Imputer: {df_mean.isnull().sum().sum()}")

In [ ]:
# Comparar distribuciones de Protein_Level antes y después
if 'Protein_Level' in df.columns:
    fig, axes = plt.subplots(1,3,figsize=(15,4))
    df['Protein_Level'].dropna().hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
    axes[0].set_title('Original (con NaN ignorados)')
    df_mean['Protein_Level'].hist(ax=axes[1], bins=30, color='orange', edgecolor='white')
    axes[1].set_title('Media Imputer (distorsiona distribución)')
    df_knn['Protein_Level'].hist(ax=axes[2], bins=30, color='seagreen', edgecolor='white')
    axes[2].set_title('KNN Imputer (preserva distribución)')
    plt.suptitle('Proteína en LCR — Impacto del Método de Imputación', y=1.02)
    plt.tight_layout()
    plt.savefig('mening_imputacion.png', dpi=100)
    plt.show()

## 6. Modelo de Clasificación

In [ ]:
# Seleccionar target
target_col = None
for c in ['Diagnosis','Risk_Level','Outcome']:
    if c in df_knn.columns:
        target_col = c
        break

print(f"Target seleccionado: {target_col}")

y_raw = df_knn[target_col].round().astype(int)
X = df_knn.drop(columns=[c for c in ['Diagnosis','Risk_Level','Outcome','Patient_ID'] 
                           if c in df_knn.columns])

# Binarizar si es necesario
if y_raw.nunique() > 2:
    y = (y_raw > y_raw.median()).astype(int)
    print(f"Target binarizado (>{y_raw.median():.1f}): {y.value_counts().to_dict()}")
else:
    y = y_raw

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:,1]

print("\n=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
# Importancia de features
imp = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10).sort_values()
imp.plot(kind='barh', figsize=(8,5), color='steelblue')
plt.title('Top 10 Features Diagnósticas — Random Forest')
plt.tight_layout()
plt.savefig('mening_importancias.png', dpi=100)
plt.show()

## ✅ Resumen
| | |
|---|---|
| Registros | 1,200 |
| Variables | 14 |
| Tipo de missingness | MAR / MNAR (estructural médico) |
| Imputación elegida | KNN Imputer (n=7) |
| Modelo | Random Forest Classifier |

**Lección clave de este dataset:**  
La **estrategia de imputación** es tan importante como el modelo. En datos médicos, imputar con la media puede hacer que una proteína elevada (señal de infección) se normalice artificialmente, destruyendo la señal diagnóstica más importante.